# Notebook 2: Signals and Drive Exports

Loads the pre-materialized Base Stack from Stage 1 and compiles all multi-scale signals directly into **42 unified GeoTIFFs** exported to Google Drive.

### Output GeoTIFFs:
1. **40 Multi-scale Stacks** (20 scales [5km to 100km] x 2 basins):
   - Standardized at EPSG:4326 geographic resolution.
   - Contains **11 bands** including cross-sectional FRIP, the GEE-computed **FRIP Mann-Kendall trend (mk_tau)**, GEDI structural signals, and environmental covariates.
2. **2 Native-scale Stacks** (1 per basin):
   - Standardized at native MODIS resolution (~463m, EPSG:4326).
   - Contains **10 bands** (GEDI signals + environmental covariates + NPP median) for high-resolution spatial modeling *without* FRIP.

### Band Structure (11 bands total for multi-scale):
- **`frip`** (1 band): Cross-sectional Spearman correlation
- **`frip_mk_tau`** (1 band): Mann-Kendall trend τ of annual FRIP across 2001–2023
- **`uoi`**, **`rh98`**, **`gedi_n`** (3 GEDI bands): Openness, height, footprint count
- **`elevation`**, **`slope`**, **`hnd`**, **`precip`**, **`clay`**, **`forest_fraction`** (6 covariate bands)

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")

# Asset paths (from NB1)
ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'

# Study regions (must match NB1)
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]

# Analysis parameters
SCALES = list(range(5000, 105000, 5000))  # 5km to 100km in 5km steps
YEARS = list(range(2001, 2024))

# Masking thresholds
FOREST_COVER_THRESHOLD = 0.95
MAX_ELEVATION = 1000
MAX_SLOPE = 10

# GEDI spatial grid (must match NB1)
GEDI_GRID_COLS = 3
GEDI_GRID_ROWS = 3
N_GEDI_TILES = GEDI_GRID_COLS * GEDI_GRID_ROWS

# Reference projection (from NB1)
_modis_col = ee.ImageCollection('MODIS/061/MOD17A3HGF').select('Npp')
MODIS_PROJ = ee.Image(_modis_col.first()).projection()
MODIS_SCALE = 463.3127165279165  # MODIS equatorial pixel size in meters



print("\u2713 Configuration loaded.")
print(f"  Scales: {SCALES[0]/1000:.0f}km - {SCALES[-1]/1000:.0f}km ({len(SCALES)} scales)")
print(f"  Basins: {[b[0] for b in BASINS]}")


In [ ]:
# =============================================================================
# BLOCK 2: IN-MEMORY COMPUTATION LOGIC
# =============================================================================

def compute_frip_mk_tau(annual_frip):
    """Computes pixel-wise Mann-Kendall trend tau across 23 years of annual FRIP.
    
    Tau = S / (n * (n - 1) / 2)
    where S = sum_{i < j} sign(x_j - x_i). For n = 23, total pairs = 253.
    """
    band_names = [f'FRIP_{y}' for y in YEARS]
    signs = []
    
    # Compute sign for all 253 unique year pairs
    for i in range(len(band_names)):
        for j in range(i + 1, len(band_names)):
            img_i = annual_frip.select(band_names[i])
            img_j = annual_frip.select(band_names[j])
            diff = img_j.subtract(img_i)
            # sign: 1 if diff > 0, -1 if diff < 0, 0 if diff == 0
            sign = diff.gt(0).subtract(diff.lt(0))
            signs.append(sign)
            
    s = ee.ImageCollection.fromImages(signs).sum()
    tau = s.divide(253).rename('frip_mk_tau')
    return tau

def build_scale_stack(base, basin_name, scale):
    """Computes all signals and covariates in memory and stacks into 11 bands."""
    base_proj = base.projection()
    
    # -------------------------------------------------------------------------
    # 1. FRIP computation (Cross-sectional + Annual + Mann-Kendall Trend)
    # -------------------------------------------------------------------------
    # Apply forest cover mask
    frip_masked = base.updateMask(
        base.select('forest_fraction').gte(FOREST_COVER_THRESHOLD)
    )
    
    # Cross-sectional FRIP
    frip_cross = frip_masked.select(['flood_freq', 'Npp_median']).reduceResolution(
        reducer=ee.Reducer.spearmansCorrelation(),
        maxPixels=262144
    ).reproject(crs='EPSG:4326', scale=scale).select('correlation').rename('frip')
    
    # Apply valid-pixels-count threshold (>10% valid pixels coverage)
    frip_cross = frip_cross.updateMask(frip_cross.mask().gt(0.1))
    
    # Annual FRIP
    def get_annual_corr(year_index):
        year_index = ee.Number(year_index)
        year = ee.Number(2001).add(year_index)
        npp_band = ee.String('NPP_').cat(year.format('%d'))
        
        corr = frip_masked.select([npp_band, 'flood_freq']).reduceResolution(
            reducer=ee.Reducer.spearmansCorrelation(),
            maxPixels=262144
        ).reproject(crs='EPSG:4326', scale=scale).select('correlation')
        
        return corr.updateMask(corr.mask().gt(0.1)).set('year', year)
    
    annual_list = ee.List.sequence(0, len(YEARS) - 1).map(get_annual_corr)
    frip_annual = ee.ImageCollection.fromImages(annual_list).toBands()
    band_names = [f'FRIP_{y}' for y in YEARS]
    frip_annual = frip_annual.rename(band_names)
    
    # Compute Mann-Kendall trend tau
    frip_mk_tau = compute_frip_mk_tau(frip_annual)
    
    # -------------------------------------------------------------------------
    # 2. GEDI signals (Masked and Aggregated)
    # -------------------------------------------------------------------------
    # Apply forest AND topo masks
    gedi_masked = base.updateMask(
        base.select('forest_fraction').gte(FOREST_COVER_THRESHOLD)
        .And(base.select('elevation').lt(MAX_ELEVATION))
        .And(base.select('slope').lt(MAX_SLOPE))
    )
    
    # GEDI assets are exported at native 25m resolution (Option 2)
    # We apply reduceResolution directly from the 25m native projection, then reproject to target scale
    uoi_agg = gedi_masked.select('GEDI_UOI').setDefaultProjection(crs='EPSG:4326', scale=25).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=262144
    ).reproject(crs='EPSG:4326', scale=scale).rename('uoi')
    
    rh98_agg = gedi_masked.select('GEDI_rh98').setDefaultProjection(crs='EPSG:4326', scale=25).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=262144
    ).reproject(crs='EPSG:4326', scale=scale).rename('rh98')
    
    n_agg = gedi_masked.select('GEDI_N').setDefaultProjection(crs='EPSG:4326', scale=25).reduceResolution(
        reducer=ee.Reducer.sum(), maxPixels=262144
    ).reproject(crs='EPSG:4326', scale=scale).rename('gedi_n')
    
    # -------------------------------------------------------------------------
    # 3. Covariates (Aggregated)
    # -------------------------------------------------------------------------
    covariates = ['elevation', 'slope', 'hnd', 'precip', 'clay', 'forest_fraction']
    covs_agg = base.select(covariates).setDefaultProjection(base_proj).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=262144
    )
    
    # -------------------------------------------------------------------------
    # 4. Concatenate into a single 11-band stack
    # -------------------------------------------------------------------------
    stack = ee.Image.cat([
        frip_cross,    # 1 band
        frip_mk_tau,   # 1 band (in GEE computed trend)
        uoi_agg,       # 1 band
        rh98_agg,      # 1 band
        n_agg,         # 1 band
        covs_agg       # 6 bands
    ]).toFloat()
    
    return stack

def build_native_stack(base):
    """Assembles high-res GEDI + covariates at native MODIS scale (~463m) without FRIP.
    
    Used for pixel-level spatial modelling of GEDI indicators at native resolution.
    """
    base_proj = base.projection()
    
    # Topo + Forest masked GEDI signals
    gedi_masked = base.updateMask(
        base.select('forest_fraction').gte(FOREST_COVER_THRESHOLD)
        .And(base.select('elevation').lt(MAX_ELEVATION))
        .And(base.select('slope').lt(MAX_SLOPE))
    )
    
    # Reduce GEDI from native 25m to MODIS scale (~463m)
    uoi = gedi_masked.select('GEDI_UOI').setDefaultProjection(crs='EPSG:4326', scale=25).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).reproject(crs=MODIS_PROJ).rename('uoi')
    
    rh98 = gedi_masked.select('GEDI_rh98').setDefaultProjection(crs='EPSG:4326', scale=25).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).reproject(crs=MODIS_PROJ).rename('rh98')
    
    gedi_n = gedi_masked.select('GEDI_N').setDefaultProjection(crs='EPSG:4326', scale=25).reduceResolution(
        reducer=ee.Reducer.sum(), maxPixels=65535
    ).reproject(crs=MODIS_PROJ).rename('gedi_n')
    
    # Covariates (including NPP median)
    covs = base.select(['elevation', 'slope', 'hnd', 'precip', 'clay', 'forest_fraction', 'Npp_median'])
    
    # Stack into 10 bands
    stack = ee.Image.cat([
        uoi, rh98, gedi_n,
        covs
    ]).toFloat()
    
    return stack

print("\u2713 In-memory computation logic loaded.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Loading Stage 1 base stack asset...")
    try:
        npp = ee.Image(f'{ASSET_ROOT}/NppStack_Congo')
        gedi_stack = ee.ImageCollection([
            ee.Image(f'{ASSET_ROOT}/GediStack_Congo_{i}') for i in range(N_GEDI_TILES)
        ]).mosaic()
        covs = ee.Image(f'{ASSET_ROOT}/CovStack_Congo')
        base = ee.Image.cat([npp, gedi_stack, covs])
        print("  \u2713 Modular NPP + gridded GediStack + Cov stacks loaded successfully")
        
        print("\nRunning multi-scale stack unit tests (using Congo at 50km)...")
        stack = build_scale_stack(base, 'Congo', 50000)
        bands = stack.bandNames().getInfo()
        
        # Test 1: Band count (11 bands expected)
        assert len(bands) == 11, f"Expected 11 bands, got {len(bands)}: {bands}"
        print("  [1/3] \u2713 Correct multi-scale band count: 11 bands assembled")
        
        # Test 2: Required bands
        required = ['frip', 'frip_mk_tau', 'uoi', 'rh98', 'gedi_n',
                    'elevation', 'slope', 'hnd', 'precip', 'clay', 'forest_fraction']
        missing = [b for b in required if b not in bands]
        assert not missing, f"Missing required bands: {missing}"
        print("  [2/3] \u2713 All 11 required signals and covariates present (including frip_mk_tau)")
        
        print("\nRunning native-scale stack unit tests (using Congo)...")
        native_stack = build_native_stack(base)
        native_bands = native_stack.bandNames().getInfo()
        
        # Test 3: Native band count (10 bands expected)
        assert len(native_bands) == 10, f"Expected 10 bands, got {len(native_bands)}: {native_bands}"
        required_native = ['uoi', 'rh98', 'gedi_n', 'elevation', 'slope', 'hnd', 'precip', 'clay', 'forest_fraction', 'Npp_median']
        missing_native = [b for b in required_native if b not in native_bands]
        assert not missing_native, f"Missing native bands: {missing_native}"
        print("  [3/3] \u2713 Correct native-scale band count and structure: 10 bands assembled")
        
        print(f"\n{'='*60}")
        print("  \u2713 ALL TESTS PASSED SUCCESSFULLY!")
        print("  Ready to launch Drive exports.")
        print(f"{'='*60}")
        
    except Exception as e:
        print(f"  \u2717 Test skipped or failed: {e}")
        print("    (This is expected if your Stage 1 BaseStack has not finished exporting yet.)")

run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 4: EXPORT TO DRIVE
# =============================================================================

def export_all_datasets(dry_run=True):
    """Launches exports for all 42 GeoTIFFs to Google Drive.
    
    - 40 Multi-scale stacks (20 scales x 2 basins)
    - 2 Native-scale stacks (1 per basin)
    Saves to Drive folder: 'DefaunationSynthesis/AnalysisStack/'
    """
    tasks = []
    
    for basin_name, basin_geom in BASINS:
        npp = ee.Image(f'{ASSET_ROOT}/NppStack_{basin_name}')
        gedi_stack = ee.ImageCollection([
            ee.Image(f'{ASSET_ROOT}/GediStack_{basin_name}_{i}') for i in range(N_GEDI_TILES)
        ]).mosaic()
        covs = ee.Image(f'{ASSET_ROOT}/CovStack_{basin_name}')
        base = ee.Image.cat([npp, gedi_stack, covs])
        
        # 1. Configure the 20 multi-scale exports (11 bands)
        for scale in SCALES:
            stack = build_scale_stack(base, basin_name, scale)
            
            task = ee.batch.Export.image.toDrive(
                image=stack,
                description=f'analysis_stack_{scale}_{basin_name}',
                folder='DefaunationSynthesis/AnalysisStack',
                fileNamePrefix=f'analysis_stack_{scale}_{basin_name}',
                region=basin_geom,
                scale=scale,
                crs='EPSG:4326',
                maxPixels=1e13
            )
            tasks.append((task, f'analysis_stack_{scale}_{basin_name}'))
            
        # 2. Configure the 1 native-scale export (10 bands)
        native_stack = build_native_stack(base)
        task_native = ee.batch.Export.image.toDrive(
            image=native_stack,
            description=f'analysis_stack_native_{basin_name}',
            folder='DefaunationSynthesis/AnalysisStack',
            fileNamePrefix=f'analysis_stack_native_{basin_name}',
            region=basin_geom,
            scale=MODIS_SCALE, # Native scale (~463m)
            crs='EPSG:4326',   # standard geodetic
            maxPixels=1e13
        )
        tasks.append((task_native, f'analysis_stack_native_{basin_name}'))
            
    print(f"\u2713 {len(tasks)} Drive export tasks configured:")
    print(f"  - 40 multi-scale exports (11 bands each)")
    print(f"  - 2 native-scale exports (10 bands each)")
    
    if dry_run:
        print("\nDRY RUN. Call export_all_datasets(dry_run=False) to launch.")
    else:
        for task, name in tasks:
            task.start()
            print(f"  \u2713 Started Drive export: {name}")
        print("\n\u2713 All 42 Drive exports started!")
        print("  Monitor at: https://code.earthengine.google.com/tasks")

export_all_datasets(dry_run=True)